🔗 [Back to Table of Contents](https://github.com/najaeda/najaeda-tutorials#-table-of-contents)

# Chapter 3: Editing a Netlist

In this chapter, we'll revisit the small full adder design from
[Chapter 1](https://colab.research.google.com/github/najaeda/najaeda-tutorials/blob/main/notebooks/01_getting_started.ipynb)
and explore how to modify it using `najaeda`'s editing API.

As a reminder, here's the schematic of the full adder:

![FullAdder](https://raw.githubusercontent.com/najaeda/najaeda-tutorials/main/images/fulladder.png)

---

## Setting Up the Environment

Let’s begin by installing **najaeda** and writing the full adder design to a local file so we can load and edit it.

In [ ]:
!pip install najaeda

In [ ]:
%%writefile fulladder.v
module halfadder(
    input a,
    input b,
    output sum,
    output carry
);
    and carry_and(carry, a, b);
    xor sum_xor(sum, a, b);
endmodule

module fulladder(
    input a,
    input b,
    input cin,
    output sum,
    output cout
);
    wire sum1, carry1, carry2;
    halfadder ha1(
        .a(a),
        .b(b),
        .sum(sum1),
        .carry(carry1)
    );
    halfadder ha2(
        .a(sum1),
        .b(cin),
        .sum(sum),
        .carry(carry2)
    );
    or cout_or(cout, carry1, carry2);
endmodule

## Loading and Visualizing the Netlist

In [ ]:

from najaeda import netlist

def display_dot(instance):
    instance.dump_full_dot(f"{instance.get_name()}.dot")
    !dot -Tpng {instance.get_name()}.dot -o {instance.get_name()}.png
    from IPython.display import Image, display
    display(Image(filename=f"{instance.get_name()}.png"))

top = netlist.load_verilog('fulladder.v')
print(f"Design name: {top.get_name()} loaded")
display_dot(top)

---

## Editing the netlist

### Renaming Instances

Let’s start by renaming the instance `ha1`.

In [ ]:
ha1 = top.get_child_instance('ha1')
print(f"Renaming instance name: {ha1.get_name()} with model: {ha1.get_model_name()} to 'halfadder1'")
ha1.set_name('halfadder1')
print(f"After renaming, renamed instance name: {ha1.get_name()} with model: {ha1.get_model_name()}")

We can safely rename `ha1` because it resides directly under the top module. No uniquification is needed in this case.

Let’s now rename it back to `ha1`:

In [ ]:
ha1.set_name('ha1')
print(f"instance name: {ha1.get_name()} with model: {ha1.get_model_name()}")

### Renaming a Lower-Level Instance

Now let's rename a child gate inside `ha1`:

In [ ]:
ha1_and = ha1.get_child_instance('carry_and')
print(f"Instance name: {ha1_and.get_name()} with model: {ha1_and.get_model_name()}")
ha1_and.set_name('and1')
print(f"After renaming, renamed instance name: {ha1_and.get_name()} with model: {ha1_and.get_model_name()}")
print(f"After renaming, {ha1.get_name()} has model {ha1.get_model_name()}")


After renaming `carry_and` to `and1`, the model of `ha1` is automatically uniquified (since `ha1` and `ha2` originally shared the same model). This ensures correctness while preserving model consistency.

Let’s visualize the netlist after the change:

In [ ]:
display_dot(top)

In [ ]:
sum1_net = top.get_net('sum1')
print(f"Net name: {sum1_net.get_name()}")
sum1_net.set_name('sum1_renamed')
print(f"After renaming, net name: {sum1_net.get_name()}")
display_dot(top)

---

## Editing the Connectivity

Let’s explore how to disconnect and reconnect terms and nets:

In [ ]:
cin = top.get_term("cin")
print(cin)
ha2 = top.get_child_instance("ha2")
print(ha2)
ha2_b = ha2.get_term("b")
print(ha2_b)
ha2_b_net = ha2_b.get_upper_net()
cin_net = cin.get_lower_net()
print(f"ha2_b_net: {ha2_b_net}, cin_net: {cin_net} are the same: {ha2_b_net == cin_net}")

### Disconnecting and Reconnecting

In [ ]:
cin.disconnect_lower_net()
display_dot(top)
cin.connect_lower_net(ha2_b_net)
display_dot(top)

### Deleting and Recreating a Net

In [ ]:
cin_net = cin.get_lower_net()
cin_net.delete()
display_dot(top)

In [ ]:
cin_net = top.create_net('cin')
cin.connect_lower_net(cin_net)
ha2_b.connect_upper_net(cin_net)
display_dot(top)

✅ This concludes **Chapter 3** of the **najaeda** tutorial. In the next chapter, we’ll load a **SystemVerilog** design and browse the **elaborated netlist**.

➡️ [**Next Chapter → Loading SystemVerilog and Browsing the Elaborated Netlist**](https://colab.research.google.com/github/najaeda/najaeda-tutorials/blob/main/notebooks/04_systemverilog_elaborated_netlist.ipynb)